# Q-values_Artificial Intelligence (CS221)_**How good is taking action a in state s? Average over where it might land you.**A value $V_\pi(s)$ rates a state. A Q-value $Q_\pi(s,a)$ rates a state and a chosen action.     It answers: if I take action $a$ now, then follow policy $\pi$ after, how much reward do I expect?---This notebook computes Q-values from a known model, one piece at a time. When we *do* have the transition probabilities and the state values, a Q-value is just a weighted average over where an action might land us. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPyGiven a fully known model — transition probabilities, rewards, and state values — we compute the **Q-value** of each (action, state) pair. We build it in three steps: (1) write down the model, (2) compute Q with a single weighted average, (3) read off the best action per state.

### Step 1 — Write down the known model`T[a, s, s']` is the probability that taking action `a` in state `s` lands us in state `s'`. `R[s']` is the reward we collect on arriving in `s'`, and `V[s']` is the already-known value of continuing from there. `gamma` discounts that future value. Here we have 2 actions and 3 states; each row of `T` sums to 1.

In [ ]:
# T[a, s, s'] transition probs; reward R(s') on arrival; known values V.
T = np.array([
    [[0.8, 0.2, 0.0], [0.0, 0.8, 0.2], [0.0, 0.0, 1.0]],   # action 0
    [[0.2, 0.8, 0.0], [0.0, 0.2, 0.8], [0.0, 0.0, 1.0]],   # action 1
])
R = np.array([0.0, 1.0, 10.0])       # reward on arriving in each state
V = np.array([5.0, 8.0, 10.0])       # known value of each state
gamma = 0.9                          # discount on future value

### Step 2 — Compute the Q-valuesThe Q-value is the expected return of taking action `a` in state `s`: a probability-weighted average over the possible next states of `R[s'] + gamma * V[s']`. First we form the per-next-state return `R + gamma * V`, then `T @ (...)` does the weighted sum over `s'` for every `(action, state)` pair at once.

In [ ]:
# Per-next-state return: immediate reward plus discounted continuation value.
future_return = R + gamma * V

# Q[a, s] = sum_s' T[a,s,s'] * (R[s'] + gamma * V[s'])
Q = T @ future_return

print("Q-values (rows = action, cols = state):")
print(np.round(Q, 2))

### Step 3 — Pick the best action per stateFor each state (each column of `Q`) the best action is the one with the highest Q-value. `argmax(axis=0)` scans down each column and returns that action index — this is the greedy policy implied by these Q-values.

In [ ]:
best = Q.argmax(axis=0)              # best action per state
print("best action per state:", best)

## Visualize the data & results_At each aisle spot, should the warehouse robot advance or retreat?_

### Step 1 — Model the warehouse aisleThree positions along an aisle: start, mid-aisle, and the shelf (where the reward of `1` lives). The robot has two actions, `advance` and `retreat`, each with its own transition probabilities — advancing tends to move it forward, retreating tends to pull it back. We reuse the same weighted-average formula to score every action.

In [ ]:
T = np.array([
    [[0.8, 0.2, 0.0], [0.0, 0.7, 0.3], [0.0, 0.0, 1.0]],   # advance
    [[1.0, 0.0, 0.0], [0.6, 0.4, 0.0], [0.0, 0.6, 0.4]],   # retreat
])
R = np.array([0.0, 0.0, 1.0])        # reward only at the shelf
V = np.array([5.0, 8.0, 10.0])
gamma = 0.9

future_return = R + gamma * V
Q = T @ future_return                # Q[action, state]

### Step 2 — Draw the Q-values as a heatmapEach row is an action (`advance` / `retreat`), each column an aisle position. Brighter cells mean a higher Q-value — a better choice there. Comparing the two rows column-by-column tells the robot whether to advance or retreat at each spot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(Q, cmap="viridis", aspect="auto")

ax.set_yticks([0, 1])
ax.set_yticklabels(["advance", "retreat"])
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["aisle start", "mid-aisle", "shelf"])

for a in range(2):
    for s in range(3):
        ax.text(s, a, round(Q[a, s], 2), ha="center", va="center", color="white")

ax.set_title("warehouse: Q(s,a) = reward + gamma V")
fig.colorbar(im, ax=ax)
plt.show()